# Demo: Detection → Bird's-Eye View → Occupancy → Decision

End-to-end pipeline on a single image. Produces:
- annotated frame (bboxes + class labels)
- bird's-eye view of the cabin floor
- footprint overlay + occupancy ratio
- final control decision (ACCEPT / BYPASS_BY_AREA / BYPASS_BY_WEIGHT)

In [ ]:
import sys, os
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import cv2, glob
import matplotlib.pyplot as plt
from src.utils.config_loader import ElevatorConfig
from src.detection.detector import YOLOv8Detector
from src.perception.homography import synthetic_homography, load_homography
from src.perception.bev import BirdsEyeView
from src.perception.occupancy import build_occupancy_estimator
from src.control.algorithm import ElevatorController

In [ ]:
WEIGHTS = '../models/weights/best.pt'   # eğittikten sonra
CONFIG  = '../configs/default.yaml'
IMAGE   = '../data/unified/test/images/' + os.listdir('../data/unified/test/images')[0]

cfg = ElevatorConfig.from_yaml(CONFIG)
img = cv2.imread(IMAGE)
h, w = img.shape[:2]
cfg = cfg.with_overrides(camera={'frame_width_px': w, 'frame_height_px': h})

# Calibrate (for now use synthetic — for real cabin run tools/calibrate_camera.py)
H = synthetic_homography(w, h, cfg.elevator.width_m, cfg.elevator.depth_m)

detector = YOLOv8Detector(
    weights_path=WEIGHTS,
    conf_threshold=cfg.thresholds.confidence_min,
    iou_threshold=cfg.thresholds.iou_min,
    target_classes=list(cfg.classes),
)
occupancy = build_occupancy_estimator(cfg.to_dict(), homography_matrix=H)
controller = ElevatorController(
    detector=detector,
    occupancy_estimator=occupancy,
    max_weight_kg=cfg.elevator.max_weight_kg,
    weight_bypass_ratio=cfg.thresholds.weight_bypass_ratio,
    area_bypass_ratio=cfg.thresholds.area_bypass_ratio,
)

WEIGHT_NOW = 250  # current cabin load in kg
result = controller.decide(WEIGHT_NOW, img)
print('DECISION   :', result.decision.value)
print('weight     :', f'{result.weight_kg} kg ({result.weight_ratio*100:.1f}%)')
print('occupancy  :', f'{(result.occupancy_ratio or 0)*100:.1f}%')
print('detections :', result.num_detections)

In [ ]:
bev = BirdsEyeView(
    homography=H,
    cabin_width_m=cfg.elevator.width_m,
    cabin_depth_m=cfg.elevator.depth_m,
    bev_size_px=cfg.camera.get('bev_resolution_px', 400),
)
detections = detector.detect(img)
viz = bev.visualize(img, detections,
                    cfg.occupancy.footprint_radius_m.to_dict(),
                    occupancy_ratio=result.occupancy_ratio)
plt.figure(figsize=(15, 6))
plt.imshow(viz[..., ::-1]); plt.axis('off'); plt.show()

## Override cabin geometry on the fly

Compare the same image under different cabin sizes (Manolya Evleri vs Cemevi).

In [ ]:
scenarios = [
    ('Default',  dict(width_m=1.4, depth_m=1.6, max_weight_kg=630)),
    ('Manolya',  dict(width_m=1.2, depth_m=1.4, max_weight_kg=450)),
    ('Cemevi',   dict(width_m=1.7, depth_m=1.8, max_weight_kg=1000)),
]
for name, params in scenarios:
    cfg2 = cfg.with_cabin(**params)
    H2 = synthetic_homography(w, h, cfg2.elevator.width_m, cfg2.elevator.depth_m)
    occ2 = build_occupancy_estimator(cfg2.to_dict(), homography_matrix=H2)
    ctrl2 = ElevatorController(
        detector=detector, occupancy_estimator=occ2,
        max_weight_kg=cfg2.elevator.max_weight_kg,
        weight_bypass_ratio=cfg2.thresholds.weight_bypass_ratio,
        area_bypass_ratio=cfg2.thresholds.area_bypass_ratio,
    )
    r = ctrl2.decide(WEIGHT_NOW, img)
    print(f'{name:>8s} | {r.decision.value:>20s} | weight={r.weight_ratio*100:5.1f}% '
          f'occ={(r.occupancy_ratio or 0)*100:5.1f}%')